# Custom event research
This notebook loads native recording samples, including the supplied synthetic fixture. Recording windows use **seconds from origin**, and timestamps retain UTC nanosecond precision. Synthetic examples do not establish field accuracy.

In [ ]:
import pandas as pd
from di_data import DIClient

di = DIClient()
datasets = await di.datasets()
pd.DataFrame(datasets)[['id', 'name', 'format', 'asset_level', 'assets', 'start', 'end']]

In [ ]:
recordings = [d for d in datasets if d['format'] == 'recording']
assert recordings, 'Import or generate a recording in Event Lab first.'
recording = recordings[0]
print(recording['name'], recording['sample_rate'], 'Hz')
print(recording['channels'])

In [ ]:
channel = next((c for c in recording['channels'] if c['kind'] == 'power'), recording['channels'][0])
page = await di.readings(recording['id'], channels=[channel['name']], start=9.5, end=10.5)
samples = di.to_frame(page)
samples.head()

In [ ]:
import matplotlib.pyplot as plt
ax = samples.plot(x='offset', y=channel['name'], figsize=(12, 4))
ax.set_xlabel('Seconds from recording origin')
ax.set_ylabel(channel['unit'])
plt.show()

## Try a custom change score
This is an exploratory difference, not a validated detector. Reset state at gaps, carry it across pages, and compare decisions only within reviewed intervals. Keep estimated onset separate from decision time.

In [ ]:
signal = samples[channel['name']]
score = signal.diff().abs()
candidates = samples.loc[score > score.quantile(0.999), ['timestamp', 'offset', channel['name']]]
candidates

In [ ]:
annotations = await di.annotations(recording['id'])
pd.DataFrame(annotations)[['kind', 'start', 'end', 'partition', 'note']]

## Stream a longer recording
Use `async for chunk in di.iter_frames(recording['id'], channels=[channel['name']], limit=100000): ...` and process each chunk before requesting the next. Avoid storing every chunk in a list. The helper never fills gaps or aggregates samples.